# 3.2 线性回归的从零开始实现

本节从零完整实现线性回归算法，覆盖数据流水线、模型、损失函数与小批量随机梯度下降优化器，仅使用张量与自动求导完成全部逻辑，帮助理解深度学习训练的底层原理。

In [ ]:
import random
import torch
from d2l import torch as d2l

## 3.2.1 生成数据集
我们构造带噪声的线性人造数据集，目标是通过有限样本恢复出模型的真实参数。数据集共1000个样本，每个样本包含2个服从标准正态分布的特征。
标签生成公式： $$y = Xw + b + \epsilon$$ 其中真实权重 $w=[2,-3.4]^\top$，偏置 $b=4.2$；噪声项 $\epsilon$ 服从均值为0、标准差为0.01的正态分布，代表观测误差。
以下代码定义数据集生成函数，并生成合成特征与标签。


In [ ]:
def synthetic_data(w, b, num_examples):  #@save
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

查看第一个样本的特征与标签，验证数据生成结果。

In [ ]:
print('features:', features[0],'\nlabel:', labels[0])

绘制第二个特征与标签的散点图，直观观察两者的线性关系。

In [ ]:
d2l.set_figsize()
d2l.plt.scatter(features[:, (1)].detach().numpy(), labels.detach().numpy(), 1);

## 3.2.2 读取数据集
训练模型时需要以小批量形式随机读取样本，充分利用 GPU 的并行计算能力。以下函数接收批量大小、特征矩阵与标签向量，打乱全部样本的顺序后，按批量大小逐批生成特征与标签。

In [ ]:
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    # 这些样本是随机读取的，没有特定的顺序
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(
            indices[i: min(i + batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

测试小批量数据读取，输出第一个批量的特征与标签形状。

In [ ]:
batch_size = 10

for X, y in data_iter(batch_size, features, labels):
    print(X, '\n', y)
    break

## 3.2.3 初始化模型参数
在优化开始前初始化模型参数：权重从均值为 0、标准差为 0.01 的正态分布中随机采样，偏置初始化为 0。同时开启`requires_grad`，让 PyTorch 自动记录并计算参数的梯度。

In [ ]:
w = torch.normal(0, 0.01, size=(2,1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

## 3.2.4 定义线性回归模型
线性回归的前向计算为：输入特征与权重做矩阵-向量乘法，再加上偏置（通过广播机制作用于每个样本）。以下函数实现线性回归的正向传播逻辑。


In [ ]:
def linreg(X, w, b):  #@save
    """线性回归模型"""
    return torch.matmul(X, w) + b

## 3.2.5 定义平方损失函数
采用平方损失作为损失函数，单样本的损失公式为： $$l^{(i)}(w,b) = \frac{1}{2}\left(\hat{y}^{(i)} - y^{(i)}\right)^2$$ 实现中会将真实标签调整为与预测值一致的形状，避免形状不匹配导致的广播错误。


In [ ]:
def squared_loss(y_hat, y):  #@save
    """均方损失"""
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

## 3.2.6 定义小批量随机梯度下降优化器
使用小批量随机梯度下降（SGD）更新模型参数，参数更新规则为： $$\theta \leftarrow \theta - \frac{\eta}{|\mathcal{B}|} \sum_{i\in\mathcal{B}} \nabla_\theta l^{(i)}$$ 其中 $\eta$ 为学习率，$\mathcal{B}$ 为当前小批量样本。函数在无梯度环境下执行参数更新，并在更新后清零梯度累积，避免影响下一次迭代。


In [ ]:
def sgd(params, lr, batch_size):  #@save
    """小批量随机梯度下降"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

## 3.2.7 模型训练
训练核心循环流程：
1.	遍历整个数据集（1个迭代周期/epoch），逐批量读取数据
2.	前向传播计算预测值与批量损失
3.	反向传播计算所有参数的梯度
4.	调用优化器更新模型参数
5.	每个epoch结束后，在全量训练集上计算平均损失并输出

设置超参数：学习率 lr=0.03，迭代周期 num_epochs=3。


In [ ]:
lr = 0.03
num_epochs = 3
net = linreg
loss = squared_loss

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y)  # X和y的小批量损失
        # 因为l形状是(batch_size,1)，而不是一个标量。l中的所有元素被加到一起，
        # 并以此计算关于[w,b]的梯度
        l.sum().backward()
        sgd([w, b], lr, batch_size)  # 使用参数的梯度更新参数
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')

**参数效果评估**

由于使用人造数据集，我们可以直接对比训练得到的参数与真实参数的误差，验证模型的学习效果。


In [ ]:
print(f'w的估计误差: {true_w - w.reshape(true_w.shape)}')
print(f'b的估计误差: {true_b - b}')